*** ***Importing modules*** ***

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping

*** ***Loading both the test and train dataset*** ***

In [2]:
train = pd.read_csv('/content/drive/MyDrive/AL ML BootCamp Data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/AL ML BootCamp Data/test.csv')

In [3]:
target = train['total_sales']
train = train.drop(columns=['total_sales'])

*** ***Combining both the train and test dataset into a single dataframe ('combined') we have*** ***

In [4]:
combined = pd.concat([train,test],ignore_index=True)

*** ***Performing some EXPLORATORY DATA ANALYSIS*** ***

In [5]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   8523 non-null   object 
 1   product_code         8523 non-null   object 
 2   product_weight_kg    6992 non-null   float64
 3   fat_content          8523 non-null   object 
 4   shelf_visibility     8523 non-null   float64
 5   product_category     8523 non-null   object 
 6   product_price        8523 non-null   float64
 7   store_code           8523 non-null   object 
 8   store_age_years      8523 non-null   int64  
 9   store_size           6113 non-null   object 
 10  store_location_tier  8523 non-null   object 
 11  store_format         8523 non-null   object 
dtypes: float64(3), int64(1), object(8)
memory usage: 799.2+ KB


*** ***Not all values are filled in both the 'product_weight_kg' and 'store_size' column*** ***

In [6]:
combined['product_weight_kg'].fillna(combined['product_weight_kg'].mean(),inplace=True)
combined['store_size'].fillna(combined['store_size'].mode()[0],inplace=True)

/tmp/ipykernel_583/631728510.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined['product_weight_kg'].fillna(combined['product_weight_kg'].mean(),inplace=True)
/tmp/ipykernel_583/631728510.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col]

In [7]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   8523 non-null   object 
 1   product_code         8523 non-null   object 
 2   product_weight_kg    8523 non-null   float64
 3   fat_content          8523 non-null   object 
 4   shelf_visibility     8523 non-null   float64
 5   product_category     8523 non-null   object 
 6   product_price        8523 non-null   float64
 7   store_code           8523 non-null   object 
 8   store_age_years      8523 non-null   int64  
 9   store_size           8523 non-null   object 
 10  store_location_tier  8523 non-null   object 
 11  store_format         8523 non-null   object 
dtypes: float64(3), int64(1), object(8)
memory usage: 799.2+ KB


*** ***All missing values were replaced*** ***

In [8]:
combined['fat_content'] = combined['fat_content'].str.lower().str.strip()
combined['product_category'] = combined['product_category'].str.lower().str.strip()

combined['product_weight_kg'] = combined.groupby('product_category')['product_weight_kg'].transform(lambda x: x.fillna(x.mean()))
store_size_mode = combined.groupby('store_format', observed=False)['store_size'].transform(lambda x: x.mode()[0] if not x.mode().empty else "Medium")
combined['store_size'] = store_size_mode

combined['shelf_visibility'] = combined['shelf_visibility'].replace(0.0, np.nan)
combined['shelf_visibility'] = combined.groupby('product_code')['shelf_visibility'].transform(lambda x: x.fillna(x.mean()))
combined['shelf_visibility'] = combined['shelf_visibility'].fillna(combined['shelf_visibility'].mean())

combined['product_type_prefix'] = combined['product_code'].str.split('-').str[1]
combined['store_code_prefix'] = combined['store_code'].str.split('-').str[1]

mean_vis = combined.groupby('store_code')['shelf_visibility'].transform('mean')
combined['visibility_store_average'] = combined['shelf_visibility'] / mean_vis

tier_mapping = {'Tier_1': 1, 'Tier_2': 2, 'Tier_3': 3}
combined['store_location_tier'] = combined['store_location_tier'].map(tier_mapping)

categorical_col = ['product_type_prefix', 'store_code_prefix', 'product_category', 'fat_content', 'store_format', 'store_size']
for col in categorical_col:
    combined[col] = combined[col].astype('category')

*** ***Splitting the 'combined' dataframe into train and test we have*** ***

In [9]:
train_df = combined.iloc[:len(train)].copy()
test_df = combined.iloc[len(train):].copy()

*** ***By adding log target to the training dataframe we have*** ***

In [10]:
train_df['target_log'] = np.log1p(target)

*** ***For target encoding and preventing data leakage we have*** ***

In [11]:
columns_to_encode = ['product_category', 'store_code']
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in columns_to_encode:
  train_df[f'{col}_encoded'] = np.nan
  test_df[f'{col}_encoded'] = np.nan

*** ***Sorting out fold values for the training set*** ***

In [12]:
for train_idx, val_idx in kf.split(train_df):
  X_tr, X_va = train_df.iloc[train_idx], train_df.iloc[val_idx]
  means = X_tr.groupby(col, observed=False)['target_log'].mean()
  train_df.iloc[val_idx, train_df.columns.get_loc(f'{col}_encoded')] = X_va[col].map(means).astype(float)

*** ***Mapping test set with global averages*** ***

In [14]:
global_means = train_df.groupby(col, observed=False)['target_log'].mean()
test_df[f'{col}_encoded'] = test_df[col].map(global_means).astype(float)

*** ***Replacing NaN with global mean averages*** ***

In [15]:
train_df[f'{col}_encoded'] = train_df[f'{col}_encoded'].fillna(train_df['target_log'].mean())
test_df[f'{col}_encoded'] = test_df[f'{col}_encoded'].fillna(train_df['target_log'].mean())

*** ***Dropping 'id','store_code' and 'product_code'*** ***

In [16]:
drop_cols = ['id', 'product_code', 'store_code']
X_train_full = train_df.drop(columns=drop_cols + ['target_log'])
X_test = test_df.drop(columns=drop_cols)
y_train_full_log = train_df['target_log']

In [23]:
import optuna
from sklearn.metrics import mean_squared_error

def objective(trial):
    params = {
        'n_estimators': 1500,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.04, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    cv_scores = []
    for train_idx, val_idx in kf.split(X_train_full):
        X_tr, X_va = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
        y_tr, y_va = y_train_full_log.iloc[train_idx], y_train_full_log.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[early_stopping(stopping_rounds=150, verbose=False)]
        )

        preds_log = model.predict(X_va)
        score = np.sqrt(mean_squared_error(y_va, preds_log))
        cv_scores.append(score)

    return np.mean(cv_scores)

*** ***Optimizing path tracking*** ****

In [25]:
study = optuna.create_study(direction='minimize')
print("Running Optuna Optimization Study...")
study.optimize(objective, n_trials=30)

print("\n--- Study Complete ---")
print("Best Log RMSE score:", study.best_value)
print("Optimized Parameters:", study.best_params)

[I 2026-09-21 10:55:18,114] A new study created in memory with name: no-name-5a56d590-91e7-482a-9551-76decb49a9f4


Running Optuna Optimization Study...


[I 2026-09-21 10:55:21,570] Trial 0 finished with value: 0.5242017781018016 and parameters: {'learning_rate': 0.014999582162363505, 'num_leaves': 26, 'min_child_samples': 60, 'feature_fraction': 0.7152184981414129, 'bagging_fraction': 0.7029206796162744, 'bagging_freq': 7}. Best is trial 0 with value: 0.5242017781018016.
[I 2026-09-21 10:55:28,178] Trial 1 finished with value: 0.5263511397145295 and parameters: {'learning_rate': 0.03406316283359198, 'num_leaves': 40, 'min_child_samples': 49, 'feature_fraction': 0.8949904874173454, 'bagging_fraction': 0.8991919201225401, 'bagging_freq': 5}. Best is trial 0 with value: 0.5242017781018016.
[I 2026-09-21 10:55:31,894] Trial 2 finished with value: 0.5245830468770991 and parameters: {'learning_rate': 0.013618125990388965, 'num_leaves': 28, 'min_child_samples': 121, 'feature_fraction': 0.6879100887817646, 'bagging_fraction': 0.6630527509305302, 'bagging_freq': 3}. Best is trial 0 with value: 0.5242017781018016.
[I 2026-09-21 10:55:36,463] Tri


--- Study Complete ---
Best Log RMSE score: 0.5221604358764476
Optimized Parameters: {'learning_rate': 0.016712714813575447, 'num_leaves': 16, 'min_child_samples': 30, 'feature_fraction': 0.6829837018688212, 'bagging_fraction': 0.6028472718704192, 'bagging_freq': 3}


In [26]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train_log, y_val_log = train_test_split(X_train_full, y_train_full_log, test_size=0.2, random_state=42)

best_params = study.best_params
best_params['n_estimators'] = 2000
best_params['random_state'] = 42
best_params['n_jobs'] = -1

final_model = LGBMRegressor(**best_params)
final_model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=[early_stopping(stopping_rounds=150, verbose=100)]
)

val_preds_log = final_model.predict(X_val)
improved_log_rmse = np.sqrt(mean_squared_error(y_val_log, val_preds_log))
print(f'Optimized Validation Log RMSE: {improved_log_rmse}')

val_preds_real = np.expm1(val_preds_log)
y_val_real = np.expm1(y_val_log)
improved_real_rmse = np.sqrt(mean_squared_error(y_val_real, val_preds_real))
print(f'Optimized Real World Scaled Validation RMSE: {improved_real_rmse:.2f}')

test_preds_log = final_model.predict(X_test)

test_preds_real = np.expm1(test_preds_log)
test_preds_real = np.clip(test_preds_real, 0, None)

store_sales_predictions = pd.DataFrame({
    'product_code': test['product_code'],
    'store_code': test['store_code'],
    'total_sales': test_preds_real
})

final_predictions = store_sales_predictions.groupby(['product_code', 'store_code'])['total_sales'].sum().reset_index()
final_predictions.to_csv('final_predictions_optimized.csv', index=False)

print("\nSuccessfully exported 'final_predictions_optimized.csv' using real sales units!")
final_predictions.head()

Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[273]	valid_0's l2: 0.266909
Optimized Validation Log RMSE: 0.5166323867182867
Optimized Real World Scaled Validation RMSE: 1135.28

Successfully exported 'final_predictions_optimized.csv' using real sales units!


,product_code,store_code,total_sales
0,PRD-00CVW9,STORE-89Z,1228.093297
1,PRD-00EIUE,STORE-7WS,4485.526434
2,PRD-00EIUE,STORE-DKU,2632.549122
3,PRD-00EIUE,STORE-JOR,390.071049
4,PRD-00EIUE,STORE-T5G,381.518870


In [27]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train_log, y_val_log = train_test_split(X_train_full, y_train_full_log, test_size=0.2, random_state=42)

best_params = study.best_params
best_params['n_estimators'] = 2000
best_params['random_state'] = 42
best_params['n_jobs'] = -1

final_model = LGBMRegressor(**best_params)
final_model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=[early_stopping(stopping_rounds=150, verbose=100)]
)

val_preds_log = final_model.predict(X_val)
improved_log_rmse = np.sqrt(mean_squared_error(y_val_log, val_preds_log))
print(f'Optimized Validation Log RMSE: {improved_log_rmse}')

val_preds_real = np.expm1(val_preds_log)
y_val_real = np.expm1(y_val_log)
improved_real_rmse = np.sqrt(mean_squared_error(y_val_real, val_preds_real))
print(f'Optimized Real World Scaled Validation RMSE: {improved_real_rmse:.2f}')

test_preds_log = final_model.predict(X_test)

test_preds_real = np.expm1(test_preds_log)
test_preds_real = np.clip(test_preds_real, 0, None)

store_sales_predictions = pd.DataFrame({
    'product_code': test['product_code'],
    'store_code': test['store_code'],
    'total_sales': test_preds_real
})

final_predictions = store_sales_predictions.groupby(['product_code', 'store_code'])['total_sales'].sum().reset_index()
final_predictions.to_csv('final_predictions_optimized.csv', index=False)

print("\nSuccessfully exported 'final_predictions_optimized.csv' using real sales units!")
final_predictions.head()

Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[273]	valid_0's l2: 0.266909
Optimized Validation Log RMSE: 0.5166323867182867
Optimized Real World Scaled Validation RMSE: 1135.28

Successfully exported 'final_predictions_optimized.csv' using real sales units!


,product_code,store_code,total_sales
0,PRD-00CVW9,STORE-89Z,1228.093297
1,PRD-00EIUE,STORE-7WS,4485.526434
2,PRD-00EIUE,STORE-DKU,2632.549122
3,PRD-00EIUE,STORE-JOR,390.071049
4,PRD-00EIUE,STORE-T5G,381.518870
